<a href="https://colab.research.google.com/github/RickPack/governed-insights-harness/blob/main/governed_insights_harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Governed Insights Harness (GIH)
### Deterministic Query Planning & Model-Risk Governance for Conversational Analytics
**Author:** Rick Pack  
**Architecture:** PydanticAI • Google Gemini 3.6 Flash • In-Memory DuckDB

---

### The Executive Problem: Semantic Ambiguity in Enterprise AI
Conversational BI and analytical chatbots frequently stall in enterprise environments due to two fundamental architectural flaws:
1. **Semantic Divergence Across Business Units:** Different departments define core entities (e.g., "customer", "revenue") with conflicting grains and business rules. An ungoverned model blends these definitions, producing misleading comparisons.
2. **Model-Risk Violations:** Large language models predict tokens—they do not calculate metrics. Allowing an LLM to perform mental math in narrative prose creates unaudited, hallucinated financial claims that fail basic compliance and risk standards.

**The Solution:** This prototype demonstrates an architectural pattern where the LLM is constrained strictly to **query planning**. The runtime routes inquiries through declarative semantic contracts, executes generated SQL deterministically inside an isolated analytical engine (DuckDB), and enforces automated governance assertion gates before any executive deliverable is emitted.

In [1]:
# 1. Install dependencies
!pip install -q pydantic-ai google-genai duckdb pandas tabulate

import os
import datetime
from typing import Any
import random
import duckdb
import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext

# 2. Authenticate Gemini via Colab Secrets
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ GEMINI_API_KEY loaded successfully from Colab Secrets.")
except Exception as e:
    print(f"⚠️ Could not load GEMINI_API_KEY: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.9/103.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.7/859.7 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.8/

---
## Architectural Pattern: Decoupled Semantic Contracts

Instead of hardcoding conflicting schema definitions into an unwieldy system prompt, this harness decouples business logic into version-controlled **Semantic Contracts**.

When presented with cross-scope questions (e.g., Department vs. Enterprise performance), the harness forces the model to evaluate the data through two explicit lenses:
* **Department Lens:** Entity Grain = `account_id` | Metric = Direct Advisory/Management Fees (`fee_amount`)
* **Enterprise Lens:** Entity Grain = `household_party_id` | Metric = Consolidated Relationship Lifetime Revenue (`total_revenue`)

By demanding a structured `ComparativeDeliverable` via Pydantic, the model cannot bypass either scope; it is forced to plan and execute queries for both.

In [2]:
def init_synthetic_database(seed: int = 42) -> duckdb.DuckDBPyConnection:
    """
    Creates an in-memory DuckDB instance with two linked tables seeded for
    reproducible semantic divergence across department and enterprise lenses.
    """
    rng = random.Random(seed)
    con = duckdb.connect(database=":memory:")

    con.execute("""
        CREATE TABLE enterprise_households (
            household_party_id VARCHAR PRIMARY KEY,
            primary_holder_name VARCHAR,
            total_revenue DOUBLE,
            product_count INTEGER,
            tenure_years INTEGER,
            relationship_tier VARCHAR,
            channel_mix VARCHAR
        );
    """)

    con.execute("""
        CREATE TABLE dept_accounts (
            account_id VARCHAR PRIMARY KEY,
            household_party_id VARCHAR,
            account_segment VARCHAR,
            fee_type VARCHAR,
            fee_amount DOUBLE,
            aum_tier VARCHAR,
            FOREIGN KEY (household_party_id) REFERENCES enterprise_households(household_party_id)
        );
    """)

    tiers = {
        "Institutional Single": {
            "count": 12,
            "products": (1, 1),
            "fee_range": (42_000, 130_000),
            "aum_tiers": ["$50M-$100M", "$100M+"],
            "fee_type": "MGMT_ADVISORY",
            "account_segment": "Institutional Contract",
            "total_revenue_factor": (1.0, 1.15),
            "tenure": (1, 6),
            "channels": ["Direct Institutional", "Prime Brokerage"],
        },
        "Multi-Product Enterprise": {
            "count": 15,
            "products": (4, 7),
            "fee_range": (8_000, 26_000),
            "aum_tiers": ["$5M-$10M", "$10M-$25M"],
            "fee_type": "MGMT_ADVISORY",
            "account_segment": "Private Wealth Sub-Account",
            "total_revenue_factor": (5.5, 9.0),
            "tenure": (7, 20),
            "channels": ["Full-Service Advisory", "Multi-Channel Wealth", "Digital + Advisor"],
        },
        "Retail Core": {
            "count": 23,
            "products": (1, 3),
            "fee_range": (1_200, 8_500),
            "aum_tiers": ["<$500K", "<$1M"],
            "fee_type": "COMMISSION",
            "account_segment": "Retail Individual Brokerage",
            "total_revenue_factor": (1.0, 1.8),
            "tenure": (1, 12),
            "channels": ["Self-Directed Digital", "Branch Referral", "Digital Only"],
        },
    }

    inst_names = [
        "Apex Global Holdings", "Meridian Capital Partners", "Summit Asset Group",
        "Vanguard Industrial Corp", "Pinnacle Fiduciary Trust", "Granite Institutional Fund",
        "Cobalt Strategic Advisors", "Ironclad Asset Management", "Northwind Capital LLC",
        "Crestview Institutional Partners", "Blackridge Fund Services", "Steelpoint Holdings"
    ]
    enterprise_names = [
        "Sterling Family Trust", "Horizon Ventures LLC", "Kensington Capital Group",
        "Lakewood Multi-Gen Trust", "Bridgeport Wealth Partners", "Clearwater Holdings",
        "Redwood Legacy Partners", "Ashford Family Office", "Mapleridge Consolidated",
        "Thornton Enterprise Trust", "Waverly Investment Group", "Cypress Multi-Asset Trust",
        "Harborview Wealth Trust", "Oakmont Capital Family", "Silverleaf Holdings LLC"
    ]
    retail_names = [f"Customer {chr(65 + i)}{j}" for i in range(5) for j in range(1, 6)]

    name_pools = {
        "Institutional Single": iter(inst_names),
        "Multi-Product Enterprise": iter(enterprise_names),
        "Retail Core": iter(retail_names),
    }

    households = []
    accounts = []
    hh_counter = 1000
    acc_counter = 100

    for tier_name, cfg in tiers.items():
        names = name_pools[tier_name]
        for _ in range(cfg["count"]):
            hh_counter += 1
            hh_id = f"HH-{hh_counter}"
            holder_name = next(names)

            num_products = rng.randint(*cfg["products"])
            tenure = rng.randint(*cfg["tenure"])
            channel = rng.choice(cfg["channels"])

            hh_total_fees = 0.0
            for _ in range(num_products):
                acc_counter += 1
                acc_id = f"ACC-{acc_counter}"
                fee = round(rng.uniform(*cfg["fee_range"]), 2)
                aum = rng.choice(cfg["aum_tiers"])
                hh_total_fees += fee

                accounts.append((
                    acc_id, hh_id, cfg["account_segment"],
                    cfg["fee_type"], fee, aum
                ))

            revenue_multiplier = rng.uniform(*cfg["total_revenue_factor"])
            total_revenue = round(hh_total_fees * revenue_multiplier, 2)

            households.append((
                hh_id, holder_name, total_revenue,
                num_products, tenure, tier_name, channel
            ))

    con.executemany("INSERT INTO enterprise_households VALUES (?, ?, ?, ?, ?, ?, ?)", households)
    con.executemany("INSERT INTO dept_accounts VALUES (?, ?, ?, ?, ?, ?)", accounts)

    hh_count = con.execute("SELECT COUNT(*) FROM enterprise_households").fetchone()[0]
    acc_count = con.execute("SELECT COUNT(*) FROM dept_accounts").fetchone()[0]
    print(f"✅ Database initialized: {hh_count} households, {acc_count} accounts seeded.")
    return con

db_con = init_synthetic_database()

✅ Database initialized: 50 households, 136 accounts seeded.


---
## Model Risk Management: The Zero-Token-Math Gate

In regulated enterprise environments, conversational outputs must be mathematically defensible. A standard LLM might look at tabular output and hallucinate rounding, extrapolations, or synthetic sums in its prose narrative.

The harness implements a **Zero-Token-Math Gate** and a **Fail-Closed Circuit Breaker**:
1. **Tool Invocation Audit:** Verifies against execution logs that queries physically ran for all required business scopes.
2. **Numeric Traceability Check:** Extracts all numerical metrics cited in the executive narrative and cross-references them against the executed DuckDB result sets.
3. **Fail-Closed Fallback:** If an unverified figure is cited and fails a retry attempt, the harness automatically suppresses the unverified prose and falls back to displaying deterministic, audited DataFrames directly to stakeholders.

In [5]:
# 1. Output Schemas
class LensAnalysis(BaseModel):
    scope: str = Field(..., description="'department' or 'enterprise'")
    primary_entity_grain: str = Field(..., description="e.g., 'account_id' or 'household_party_id'")
    summary: str = Field(..., description="Key qualities of top customers under this lens (tenure, tier, product mix)")
    top_segments: list[str] = Field(..., description="Distinct segments surfaced")

class ComparativeDeliverable(BaseModel):
    department_lens: LensAnalysis
    enterprise_lens: LensAnalysis
    reconciliation_memo: str = Field(
        ...,
        description="Plain-English explanation of why top customers differ across department vs. enterprise definitions."
    )
    cited_metrics: list[float] = Field(
        default_factory=list,
        description="Every numerical metric cited in your narrative summaries that must be verified against query results."
    )

# 2. Runtime Context & Tools
class HarnessContext:
    def __init__(self, db: duckdb.DuckDBPyConnection):
        self.db = db
        self.tool_log: list[dict[str, Any]] = []
        self.raw_dataframes: list[pd.DataFrame] = []

agent = Agent(
    "google:gemini-3.6-flash",
    deps_type=HarnessContext,
    output_type=ComparativeDeliverable,
    system_prompt=(
        "You are an enterprise query planning and analytical governance agent. "
        "You NEVER perform mental arithmetic. All metric claims must derive strictly "
        "from SQL execution. When asked cross-scope questions, you MUST examine both the "
        "'department' and 'enterprise' scopes using the run_contract_query tool. "
        "Always write analytical queries with aggregations (e.g., GROUP BY, ORDER BY, LIMIT) "
        "rather than dumping raw tables. Execute exactly one targeted analytical aggregation "
        "query per scope (department and enterprise) using GROUP BY, ORDER BY, and LIMIT. "
        "Do not run exploratory SELECT * queries. If a query returns a database error, "
        "you may issue one corrected analytical query.Report all raw numerical metric "
        "figures cited in your text inside the cited_metrics array for automated audit "
        "verification."
    )
)

@agent.tool
def list_contracts(ctx: RunContext[HarnessContext]) -> list[dict[str, str]]:
    """Returns active semantic contracts, entity grains, tables, and target metrics."""
    return [
        {
            "scope": "department",
            "table": "dept_accounts",
            "entity_grain": "account_id",
            "metric": "fee_amount (Management & Advisory Fees)",
            "columns": "account_id, household_party_id, account_segment, fee_type, fee_amount, aum_tier",
            "description": "Departmental operational revenue measured at the single account/contract grain."
        },
        {
            "scope": "enterprise",
            "table": "enterprise_households",
            "entity_grain": "household_party_id",
            "metric": "total_revenue (Consolidated Firm-Wide Revenue)",
            "columns": "household_party_id, primary_holder_name, total_revenue, product_count, tenure_years, relationship_tier, channel_mix",
            "description": "Consolidated enterprise relationship revenue aggregated across all product lines per household."
        }
    ]

@agent.tool
def run_contract_query(ctx: RunContext[HarnessContext], scope: str, sql_query: str) -> str:
    """Executes SQL against DuckDB. Intercepts errors and returns them as observations for self-correction."""
    clean_scope = scope.lower().strip()
    try:
        df = ctx.deps.db.execute(sql_query).df()
        ctx.deps.tool_log.append({"scope": clean_scope, "sql": sql_query, "status": "SUCCESS"})
        ctx.deps.raw_dataframes.append(df)
        return df.to_string(index=False)
    except Exception as sql_err:
        ctx.deps.tool_log.append({"scope": clean_scope, "sql": sql_query, "status": f"SQL_ERROR: {sql_err}"})
        return f"Database Error: {sql_err}. Please inspect table schemas and generate corrected SQL."

# 3. Governance Verification Gates
def verify_tool_coverage(tool_log: list[dict[str, Any]]) -> None:
    """Asserts that queries physically ran successfully for both business scopes."""
    invoked_scopes = {call["scope"] for call in tool_log if call["status"] == "SUCCESS"}
    required = {"department", "enterprise"}
    missing = required - invoked_scopes
    if missing:
        raise AssertionError(f"Tool Coverage Gate Failure: Missing successful query execution for scope(s): {missing}")

def verify_zero_token_math(cited_metrics: list[float], raw_dfs: list[pd.DataFrame]) -> None:
    """Asserts that all numerical claims exist in the executed query result sets."""
    if not cited_metrics:
        return

    executed_values = set()
    for df in raw_dfs:
        for col in df.select_dtypes(include=["number"]).columns:
            for val in df[col].dropna():
                executed_values.add(round(float(val), 2))
                executed_values.add(round(float(val), 0))

    unverified = [
        m for m in cited_metrics
        if round(float(m), 2) not in executed_values and round(float(m), 0) not in executed_values
    ]
    if unverified:
        raise AssertionError(
            f"Zero-Token-Math Gate Failure: Figures {unverified} cannot be traced to executed SQL results."
        )

def print_audit_log(prompt: str, ctx: HarnessContext, status: str) -> None:
    """Emits a structured, auditable execution trace."""
    print("\n" + "=" * 80)
    print(f"AUDIT LOG: EXECUTION TRACE — {datetime.datetime.now(datetime.timezone.utc).isoformat()}")
    print("=" * 80)
    print(f"Prompt: {prompt}")
    print(f"Governance Status: {status}\n")
    print("Executed Queries:")
    for idx, entry in enumerate(ctx.tool_log, 1):
        print(f"  [{idx}] Scope: {entry['scope'].upper()} | Status: {entry['status']}")
        print(f"      SQL: {entry['sql']}")
    print("=" * 80 + "\n")

async def run_governed_inquiry(prompt: str, ctx: HarnessContext, max_retries: int = 1):
    """Executes inquiry asynchronously, enforces verification gates, and fails closed upon error."""
    current_prompt = prompt

    for attempt in range(max_retries + 1):
        try:
            ctx.tool_log.clear()
            ctx.raw_dataframes.clear()

            # Native async execution
            result = await agent.run(current_prompt, deps=ctx)

            verify_tool_coverage(ctx.tool_log)
            verify_zero_token_math(result.output.cited_metrics, ctx.raw_dataframes)

            print_audit_log(prompt, ctx, status="PASSED_ALL_GATES")
            return result.output

        except Exception as gate_err:
            if attempt < max_retries:
                current_prompt = f"{prompt}\n\n[PREVIOUS RUN REJECTED BY GOVERNANCE GATE]: {str(gate_err)}"
                continue

            # Fail-Closed Fallback
            print_audit_log(prompt, ctx, status=f"FAILED: {gate_err}")
            print("⚠️  [GOVERNANCE ALERT] Automated verification could not certify claims.")
            print(f"Reason: {gate_err}")
            print("Action: Narrative memo suppressed. Displaying deterministic query outputs below.\n")

            for idx, df in enumerate(ctx.raw_dataframes, 1):
                print(f"--- Deterministic Query Result [{idx}] ---")
                print(df.to_string(index=False))
                print()
            return None

---
## Executive Deliverable

The cell below renders the governed output for a business audience. Metrics are formatted for readability, the semantic reconciliation memo explains why the two lenses disagree, and the audit trail is one click away — every figure in the narrative traces back to an executed query.

In [7]:
"""
EXECUTIVE RENDERING CELL
Governed Insights Harness — stakeholder presentation layer.
"""

from IPython.display import display, HTML
import re

# ==============================================================================
# FORMATTING UTILITIES
# ==============================================================================

def format_numbers_in_text(text: str) -> str:
    """
    Formats numbers in narrative text for executive readability:
    - Adds thousands separators (95790.07 -> 95,790.1)
    - Rounds floats to 1 decimal place
    - Preserves 4-digit calendar years (1900-2099)
    - Strips characters that break Markdown rendering
    """
    text = text.replace("$", "").replace("_", " ")

    def _fmt(match):
        raw = match.group(0)
        try:
            num = float(raw)
            if num == int(num) and "." not in raw:
                val = int(num)
                if 1900 <= val <= 2099:
                    return raw
                return f"{val:,}"
            else:
                return f"{num:,.1f}"
        except ValueError:
            return raw

    return re.sub(r'\b\d[\d,]*\.?\d*\b', _fmt, text)


def styled_card(title: str, content: str, accent: str = "#0A6640") -> str:
    """Returns an HTML card with a left accent border."""
    return f"""
    <div style="
        border-left: 4px solid {accent};
        padding: 14px 18px;
        margin: 12px 0;
        background: #f8f9fa;
        border-radius: 0 6px 6px 0;
        font-size: 14px;
        line-height: 1.6;
        color: #1a1a1a;
    ">
        <div style="font-weight: 600; font-size: 15px; margin-bottom: 6px; color: {accent};">
            {title}
        </div>
        {content}
    </div>
    """


def status_badge(passed: bool, metric_count: int = 0, reason: str = "") -> str:
    """Returns a colored governance pass/fail badge."""
    if passed:
        color, bg, icon, label = "#0A6640", "#e6f4ed", "✓", "ALL GATES PASSED"
        detail = f"{metric_count} metrics verified against SQL results"
    else:
        color, bg, icon, label = "#c62828", "#fce8e8", "✗", "VERIFICATION FAILED"
        detail = reason or "Execution intercepted by governance gate"

    return f"""
    <div style="
        display: inline-block;
        padding: 8px 16px;
        background: {bg};
        border: 1px solid {color};
        border-radius: 6px;
        font-weight: 600;
        font-size: 14px;
        color: {color};
        margin: 8px 0;
    ">
        {icon} {label} &nbsp;·&nbsp; {detail}
    </div>
    """


def audit_trail(tool_log: list) -> str:
    """Returns a collapsible HTML block showing every executed query."""
    rows = ""
    for idx, entry in enumerate(tool_log, 1):
        rows += f"""
        <div style="margin: 10px 0; padding: 10px; background: #fff;
                    border: 1px solid #ddd; border-radius: 4px;">
            <div style="font-weight: 600; font-size: 13px; color: #0A6640;">
                Query {idx} &nbsp;·&nbsp; Scope: {entry['scope'].upper()} &nbsp;·&nbsp; {entry['status']}
            </div>
            <pre style="margin: 6px 0 0 0; font-size: 12px; white-space: pre-wrap;
                        color: #333; font-family: monospace;">{entry['sql']}</pre>
        </div>
        """
    return f"""
    <details style="margin-top: 20px;">
        <summary style="cursor: pointer; font-weight: 600; color: #0A6640;
                        font-size: 14px; padding: 8px 0;">
            View Audit Trail ({len(tool_log)} queries executed)
        </summary>
        <div style="padding: 10px 0;">{rows}</div>
    </details>
    """


# ==============================================================================
# 1. INITIALIZE & RUN
# ==============================================================================

runtime_ctx = HarnessContext(db=db_con)

executive_query = (
    "What qualities describe the customers who per customer produce the most revenue "
    "for our department vs. the entire business?"
)

print(f"Initiating Governed Analytics Loop...\nQuery: '{executive_query}'\n")

deliverable = await run_governed_inquiry(executive_query, runtime_ctx)

# ==============================================================================
# 2. RENDER EXECUTIVE DELIVERABLE
# ==============================================================================

if deliverable:
    dept = deliverable.department_lens
    ent = deliverable.enterprise_lens

    # --- Header ---
    display(HTML("""
    <div style="border-bottom: 3px solid #0A6640; padding-bottom: 8px; margin-bottom: 20px;">
        <h2 style="color: #0A6640; margin: 0;">Executive Decision Deliverable</h2>
        <span style="color: #666; font-size: 14px;">
            Dual-Lens Customer Profile · Governed Insights Harness
        </span>
    </div>
    """))

    # --- Key Finding Callout ---
    display(HTML("""
    <div style="
        background: #fff8e1;
        border: 1px solid #ffc107;
        padding: 14px 18px;
        border-radius: 6px;
        margin-bottom: 16px;
        font-size: 15px;
        line-height: 1.5;
        color: #1a1a1a;
    ">
        <strong>Key Finding:</strong> The two lenses surface entirely different customer
        populations. Department-level analysis favors high-fee institutional contracts;
        enterprise-level analysis favors multi-product households with deep relationship
        breadth. Both answers are correct within their own business definition.
    </div>
    """))

    # --- Comparison Table ---
    display(HTML(f"""
    <table style="width: 100%; border-collapse: collapse; font-size: 14px; margin: 16px 0;">
        <thead>
            <tr style="background: #0A6640; color: white;">
                <th style="padding: 10px 14px; text-align: left; width: 25%;">Dimension</th>
                <th style="padding: 10px 14px; text-align: left; width: 37%;">Department Lens</th>
                <th style="padding: 10px 14px; text-align: left; width: 37%;">Enterprise Lens</th>
            </tr>
        </thead>
        <tbody>
            <tr style="background: #f8f9fa;">
                <td style="padding: 10px 14px; font-weight: 600;">Entity Grain</td>
                <td style="padding: 10px 14px;"><code>{dept.primary_entity_grain}</code></td>
                <td style="padding: 10px 14px;"><code>{ent.primary_entity_grain}</code></td>
            </tr>
            <tr>
                <td style="padding: 10px 14px; font-weight: 600;">Top Segments</td>
                <td style="padding: 10px 14px;">{format_numbers_in_text(', '.join(dept.top_segments))}</td>
                <td style="padding: 10px 14px;">{format_numbers_in_text(', '.join(ent.top_segments))}</td>
            </tr>
        </tbody>
    </table>
    """))

    # --- Lens Analysis Cards ---
    display(HTML(styled_card(
        "Department Lens Analysis",
        format_numbers_in_text(dept.summary),
        accent="#1565C0"
    )))

    display(HTML(styled_card(
        "Enterprise Lens Analysis",
        format_numbers_in_text(ent.summary),
        accent="#0A6640"
    )))

    # --- Reconciliation Memo ---
    display(HTML("""
    <h3 style="color: #333; margin-top: 24px; margin-bottom: 4px;">
        Semantic Reconciliation Memo
    </h3>
    """))

    display(HTML(styled_card(
        "Why the Top Customers Differ Across Lenses",
        format_numbers_in_text(deliverable.reconciliation_memo),
        accent="#6A1B9A"
    )))

    # --- Governance Status ---
    display(HTML("""
    <h3 style="color: #333; margin-top: 24px; margin-bottom: 4px;">
        Model-Risk Validation
    </h3>
    """))

    display(HTML(status_badge(
        passed=True,
        metric_count=len(deliverable.cited_metrics)
    )))

    # --- Audit Trail ---
    display(HTML(audit_trail(runtime_ctx.tool_log)))

else:
    # --- Fail-Closed Executive Intercept ---
    display(HTML("""
    <div style="border-bottom: 3px solid #c62828; padding-bottom: 8px; margin-bottom: 20px;">
        <h2 style="color: #c62828; margin: 0;">Execution Halted (Fail-Closed)</h2>
        <span style="color: #666; font-size: 14px;">
            Governed Insights Harness · Model-Risk Policy Enforcement
        </span>
    </div>
    """))

    display(HTML(status_badge(passed=False)))

    display(HTML(styled_card(
        "Safety Intercept Triggered",
        "Automated governance could not verify all narrative claims against executed "
        "query results. Narrative output suppressed to protect reporting integrity. "
        "Deterministic query results are displayed in the audit log above.",
        accent="#c62828"
    )))

    display(HTML(audit_trail(runtime_ctx.tool_log)))

Initiating Governed Analytics Loop...
Query: 'What qualities describe the customers who per customer produce the most revenue for our department vs. the entire business?'


AUDIT LOG: EXECUTION TRACE — 2026-09-15T11:11:45.935668+00:00
Prompt: What qualities describe the customers who per customer produce the most revenue for our department vs. the entire business?
Governance Status: PASSED_ALL_GATES

Executed Queries:
  [1] Scope: DEPARTMENT | Status: SUCCESS
      SQL: SELECT 
    account_segment,
    aum_tier,
    fee_type,
    COUNT(DISTINCT account_id) AS account_count,
    SUM(fee_amount) AS total_department_revenue,
    AVG(fee_amount) AS avg_fee_per_account
FROM dept_accounts
GROUP BY account_segment, aum_tier, fee_type
ORDER BY avg_fee_per_account DESC
LIMIT 10;
  [2] Scope: ENTERPRISE | Status: SUCCESS
      SQL: SELECT 
    relationship_tier,
    channel_mix,
    AVG(tenure_years) AS avg_tenure_years,
    AVG(product_count) AS avg_product_count,
    COUNT(DISTINCT household_p

Dimension,Department Lens,Enterprise Lens
Entity Grain,account_id,household_party_id
Top Segments,"Institutional Contract (100M+ AUM), Institutional Contract (50M-100M AUM), Private Wealth Sub-Account (5M-10M AUM)","Multi-Product Enterprise (Full-Service Advisory), Multi-Product Enterprise (Digital + Advisor), Multi-Product Enterprise (Multi-Channel Wealth)"


---
## Strategic Roadmap & Future Extensions

This prototype demonstrates the foundational execution loop. In a production enterprise deployment, the harness extends across four key architectural areas:

1. **Enterprise Semantic Layer Integration:** Moving from in-notebook contract definitions to automated ingestion of enterprise semantic definitions (e.g., dbt semantic layer, Cube, or enterprise data catalogs).
2. **Advanced Model-Risk Assertion Gates:**
   * *Revenue Hierarchy Sanity:* Asserting that departmental sub-totals do not exceed consolidated parent totals.
   * *Denominator Grain Validation:* Verifying that ratios divide by distinct entity counts (`COUNT(DISTINCT account_id)`) rather than raw transaction rows.
3. **Automated Evaluation Benchmarks:** A regression test suite of 50+ golden business-unit queries to continuously benchmark tool-calling accuracy, join precision, and schema drift across LLM version updates.
4. **Role-Based Tenant Access:** Dynamic contract filtering based on user authentication, ensuring departmental analysts only execute queries against authorized data grains.